# P2 training only: Prithvi + U-Net + WCE

Notebook này chỉ phụ trách train/resume P2 và chọn checkpoint. Nó không tạo
bản đồ, không benchmark inference và không đối chiếu điểm cũ. Notebook P2 đã
chạy đầy đủ trước đây được giữ nguyên như evidence snapshot.

Mặc định `RUN_TRAINING=False` để tránh vô tình tiêu tốn GPU. Chỉ bật sau khi
đã duyệt run directory, commit và dữ liệu.

## 1. Mount Drive và checkout mã nguồn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
from pathlib import Path

REPO_URL = 'https://github.com/hongphuc-maker/xuan-thuy-prithvi-segmentation.git'
REPO_REF = 'main'  # Replace with an approved commit SHA before a scientific run.
REPO_ROOT = Path('/content/xuan-thuy-prithvi-segmentation')

if not (REPO_ROOT / '.git').exists():
    subprocess.check_call(['git', 'clone', '--filter=blob:none', '--no-checkout', REPO_URL, str(REPO_ROOT)])
else:
    dirty = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'status', '--porcelain'], text=True).strip()
    if dirty:
        raise RuntimeError('The Colab repository copy has local changes; use a fresh runtime or review them.')

subprocess.check_call(['git', '-C', str(REPO_ROOT), 'fetch', '--depth=1', 'origin', REPO_REF])
subprocess.check_call(['git', '-C', str(REPO_ROOT), 'checkout', '--detach', 'FETCH_HEAD'])

REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
print('Repository commit:', REPO_COMMIT)


## 2. Cài môi trường P2 theo version đã khóa

In [ ]:
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'terratorch==1.2.12', 'torchgeo==0.9.0', 'timm==1.0.30',
    'segmentation-models-pytorch==0.5.0', 'huggingface-hub==1.33.0',
    'einops==0.8.2', 'rasterio==1.5.2', 'pandas==2.2.3',
    'matplotlib>=3.8,<4', 'PyYAML>=6,<7', 'geopandas>=1,<2', 'tqdm>=4.66,<5',
    'git+https://github.com/hongphuc-maker/xuan-thuy-dmi-segmentation.git@0775770ae87b83b7d2982ed727226128685f3fba'
])

import sys
P2_SOURCE = REPO_ROOT / 'reference' / 'wce_completed'
sys.path.insert(0, str(P2_SOURCE))
print('P2 source:', P2_SOURCE)


## 3. Khóa đường dẫn dữ liệu và run training

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Prithvi_Colab')
DATA_ROOT = DRIVE_ROOT / 'data' / 'training_p2'
RUN_ID = 'P2_multitemporal6_sep21_UNet_WCE_REPRO'
RUN_ROOT = DRIVE_ROOT / 'runs' / RUN_ID
CONFIG_PATH = REPO_ROOT / 'configs' / 'experiments' / 'p2_prithvi_unet_wce_completed.json'

print('DATA_ROOT:', DATA_ROOT)
print('RUN_ROOT:', RUN_ROOT)
print('CONFIG_PATH:', CONFIG_PATH)
assert CONFIG_PATH.is_file()


## 4. Audit contract, split và dữ liệu

In [ ]:
import json
from IPython.display import display
from prithvi_xtseg.training import open_experiment

experiment = open_experiment(CONFIG_PATH, DATA_ROOT, RUN_ROOT)
display(experiment.scene.audit)
assert experiment.scene.audit['shared_input_pixels'] == 0
print('Train patches:', len(experiment.manifest))
print('Validation windows:', len(experiment.val))


## 5. GPU smoke test

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select Runtime > Change runtime type > GPU')
model = experiment.load_model(pretrained=not (RUN_ROOT / 'latest_resume.pt').exists())
smoke = experiment.smoke_test()
display(smoke)


## 6. Train hoặc resume P2

In [ ]:
RUN_TRAINING = False

if not RUN_TRAINING:
    print('Training is disabled. Review the paths and set RUN_TRAINING=True after approval.')
else:
    training_result = experiment.train()
    display(training_result)


## 7. Đọc model-selection artifacts, không tạo map

In [ ]:
for filename in ['TRAINING_PROGRESS.json', 'TRAINING_COMPLETE.json', 'MODEL_SELECTION.json']:
    path = RUN_ROOT / filename
    if path.exists():
        print('\n', filename)
        display(json.loads(path.read_text()))

print('This notebook stops before map inference by design.')
